# CMP4509 — Kayıtlı MNIST sonuçlarını denetle ve görselleri yeniden üret

Bu notebook, **önceden kaydedilmiş dokuz modelin sonuç tablosunu** okuyarak
ortalama ± örnek standart sapmayı ve rapor şekillerini yeniden üretir.
**Model eğitmez, checkpoint yüklemez, MNIST indirmez ve saldırı çalıştırmaz.**
Aritmetik/biçim denetimi, tarihî deneyin bağımsız olarak yeniden yürütülmesi anlamına gelmez.

Notebook gereken dört dosyayı kendi içinde taşır; depo klonlamak veya GPU kullanmak gerekmez.
Colab, Kaggle veya yerel Jupyter'da hücreleri sırayla çalıştırın.
Yalnız eksik NumPy/Matplotlib paketlerinin ilk kurulumu internet gerektirebilir.

- Ana test: dokuz kayıt, üç yöntem × üç eğitim seed'i, kayıt başına 10.000 MNIST test görüntüsü.
- Belgelenen saldırı: L∞, ε=0.3, α=0.01, PGD100 × **5** rastgele başlangıç.
  Bu ayarlar özgün CSV'nin sütunları değildir; ayrı protokol kaydından gelir.
- Ek validation: bildirilen aynı 512 görüntü için PGD100 × **20** ve full standard AutoAttack.
  Yüzdeler yazarın yuvarlatılmış beyanıdır; ham loglar ve subset indeksleri mevcut değildir.
  Test ile validation sonuçları birleştirilmez.
- Checkpoint SHA-256 alanlarının biçimi denetlenir; model dosyaları bulunmadığından
  hash'lerin gerçek checkpoint içerikleriyle eşleşmesi doğrulanamaz.

Yeni eğitim/saldırı deneyleri için ayrı `01_mnist_reproduction.ipynb` notebook'unu kullanın.
Public proje: https://github.com/emirbakac/cmp4509-madry-mnist


## 1 · Yalnız eksik analiz paketlerini kur

Kurulu paketler korunur. Bu analiz için PyTorch, torchvision veya AutoAttack gerekmez.


In [ ]:
import importlib.util
import subprocess
import sys

for module, requirement in {
    "numpy": "numpy>=1.26,<3",
    "matplotlib": "matplotlib>=3.8,<4",
}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", requirement])

print("Python:", sys.version.split()[0])
print("Analiz ortamı hazır; GPU gerekmiyor.")


## 2 · Gömülü kaynakların birebir kopyasını ayrı analiz klasörüne aç

Çalışma dizini platforma göre `/kaggle/working`, `/content` veya mevcut klasör altında
oluşturulur. Her gömülü kaynak sürümünün içerik hash'inden türeyen ayrı bir snapshot'ı vardır.
Aynı sürümü yeniden çalıştırmak kaynakları değiştirmez. Mevcut snapshot dosyası elle
değiştirilmişse işlem durur; kullanıcı dosyalarının üzerine sessizce yazılmaz.
Hesaplanan istatistik ve şekiller ise aynı snapshot içinde yeniden oluşturulabilir.


In [ ]:
from pathlib import Path
import base64
import hashlib
import json
import os

if "_RECORDED_ANALYSIS_BASE" not in globals():
    _RECORDED_ANALYSIS_BASE = (
        Path("/kaggle/working") if Path("/kaggle/working").is_dir()
        else Path("/content") if Path("/content").is_dir()
        else Path.cwd()
    )
ANALYSIS_ROOT = _RECORDED_ANALYSIS_BASE / "madry_mnist_recorded_analysis"


In [ ]:
EMBEDDED = {
  "results/FINAL_TEST_ALL_MODELS.csv": {
    "base64": "bWV0aG9kLHNlZWQsYmVzdF9lcG9jaCxjbGVhbl9hY2N1cmFjeSxmZ3NtX2FjY3VyYWN5LHBnZF9hY2N1cmFjeSx0ZXN0X2ltYWdlcyxjaGVja3BvaW50X3NoYTI1NgpCYXNlbGluZSwwLDEwLDAuOTkwNywwLjEyMzgsMC4wLDEwMDAwLGI4ODVhNjgxN2RjNjE4ZmE5N2I2OWNmMGRkMjBlMjk2ZWYyYzQyY2UxYzNjNmE3NzU2MjA5YWNlYWI4OGIzYzIKQmFzZWxpbmUsMSw4LDAuOTkwNSwwLjEyMDMsMC4wLDEwMDAwLDc4NDZkZWJkYzRiZWZhNDE1OTU4Y2EyN2Y1Mjg0ZjZmMmJiYzFjNTk1Mjc0NzkwNDJiMzRkNjMyMDhmMzg5ZDMKQmFzZWxpbmUsMiwxMCwwLjk4OTcsMC4xMDA3LDAuMCwxMDAwMCwzYjI2MGEzNjUzNWM1ZDI5Y2ZhOGI4MDVhYmQ1M2ZmNTRkZWVkZjQ4Njk5ZWNlZDY0NDVkZTU0YmE3ODQ5ZDE4ClBHRDQwLDAsMTAsMC45MTY2LDAuODA1NCwwLjcxMTUsMTAwMDAsMjhjMWE3MzRlYmU5YzM1OGM2MWZlN2M4ZWZkM2QwZjFiMzRjYjgyZWFjMDJlYzExY2U2NzFhOTFmMTdhOGZkNQpQR0Q0MCwxLDEsMC4xMTM1LDAuMTEzNSwwLjExMzUsMTAwMDAsODJkZGQxMDBlNjQ4ODQwZTNlZDNjYmE5NGFmNDVhYmVjM2NhMjA0ZjYzMDI0ZjIzOTgzZjk2Yzk0NDMyMDY5NwpQR0Q0MCwyLDEsMC4xMTM1LDAuMTEzNSwwLjExMzUsMTAwMDAsMTVhOTgwYWNlYmEwMjBhZjdiYjk0ODdlZTdiMmFkNzRkNzc5OTBkMDBkNDhhYTEzYTY0MWY3Nzg4NzZmYzk4MQpDdXJyaWN1bHVtLDAsMTAsMC45NzUsMC45MzIyLDAuODc2MiwxMDAwMCw0ZjM4NTQwNzhmMmY0YjA2MDc2NGVjN2NhYjYwN2RiZTYyZGM2OGMxOWRjNTI0ZTk3ZmQ0ODE1MTNjMzczY2U3CkN1cnJpY3VsdW0sMSwxMCwwLjk4MDEsMC45NDQyLDAuODkxNCwxMDAwMCwwNmI3ODY0MjkxMGJhZmFkMDM2NmZjNDgxNzk4ZjU2MjM0Y2VjOWJkZjhhMzQwNGMxZmIxYzUwYjI1NTlhMzMwCkN1cnJpY3VsdW0sMiwxMCwwLjk4MDMsMC45NDA5LDAuODc2MywxMDAwMCxlMjNiNWIzOTA0MDM0OTllOGZiNTMzYThmMDZiOGM1MDdmMjE0ZjY0NTViZTk5NTBhNGM3YmM0YWZhYjFjN2FkCg==",
    "sha256": "654e9696f9a11c86e514b44ea9bd9d38fcfb81dabb281d7482b25f8e7de5d345"
  },
  "results/FINAL_TEST_PROTOCOL.json": {
    "base64": "ewogICJtZWFzdXJlbWVudF9maWxlIjogIkZJTkFMX1RFU1RfQUxMX01PREVMUy5jc3YiLAogICJtZWFzdXJlbWVudF9zaGEyNTYiOiAiNjU0ZTk2OTZmOWExMWM4NmU1MTRiNDRlYTliZDlkMzhmY2ZiODFkYWJiMjgxZDc0ODJiMjVmOGU3ZGU1ZDM0NSIsCiAgInByb3RvY29sX3Byb3ZlbmFuY2UiOiAiRnJvemVuIHByb3RvY29sIHJlcG9ydGVkIGJ5IHRoZSBhdXRob3IgYW5kIHN1cHBsaWVkIG1hbnVzY3JpcHQuIFRoZSBjb21wYWN0IG9yaWdpbmFsIENTViBpdHNlbGYgZG9lcyBub3QgZW5jb2RlIGF0dGFjayBzZXR0aW5ncywgZXhlY3V0aW9uIGVudmlyb25tZW50LCBkYXRhIGluZGljZXMsIG9yIGNvZGUgcmV2aXNpb24uIiwKICAiZGF0YXNldCI6ICJNTklTVCIsICJzcGxpdCI6ICJ0ZXN0IiwgIm5fZXhhbXBsZXNfcGVyX2NoZWNrcG9pbnQiOiAxMDAwMCwKICAibm9ybSI6ICJMaW5mIiwgImVwc2lsb24iOiAwLjMsICJwaXhlbF9yYW5nZSI6IFswLCAxXSwKICAicGdkX3N0ZXBzIjogMTAwLCAicGdkX2FscGhhIjogMC4wMSwgInBnZF9yYW5kb21fcmVzdGFydHMiOiA1LAogICJhdHRhY2tfb2JqZWN0aXZlIjogInVudGFyZ2V0ZWQgY3Jvc3MtZW50cm9weSIsCiAgInRyYWluaW5nX3NlZWRzIjogWzAsIDEsIDJdLAogICJyZXBvcnRlZF9jaGVja3BvaW50X3NlbGVjdGlvbiI6ICJCZXN0IGNsZWFuIHZhbGlkYXRpb24gZm9yIGJhc2VsaW5lLCBiZXN0IFBHRCB2YWxpZGF0aW9uIGZvciBhZHZlcnNhcmlhbCBjb25kaXRpb25zOyBmaXJzdCBlcG9jaCB3aW5zIHRpZXMgaW4gYXVkaXRlZCBjb2RlLiIsCiAgImNoZWNrcG9pbnRfZmlsZXNfYXZhaWxhYmxlX3RvX2F1ZGl0IjogZmFsc2UsCiAgImhpc3RvcmljYWxfZXhlY3V0aW9uX3JlcGxheWVkIjogZmFsc2UsCiAgImhpc3RvcmljYWxfY29kZV9yZXZpc2lvbiI6IG51bGwsCiAgImhpc3RvcmljYWxfZXZhbF9ybmdfc2VlZCI6IG51bGwsCiAgImF1ZGl0X3N0YXR1cyI6ICJOaW5lIHNvdXJjZSByb3dzLCBhbGwgYWdncmVnYXRlcywgZGVub21pbmF0b3JzLCByYW5nZXMsIGFuZCBoYXNoIHN5bnRheCBjaGVja2VkOyBhY3R1YWwgdHJhaW5lZCBjaGVja3BvaW50cyBhbmQgaGlzdG9yaWNhbCBldmFsdWF0aW9uIHByb2Nlc3Mgbm90IGluZGVwZW5kZW50bHkgcmVwbGF5ZWQuIgp9Cg==",
    "sha256": "da95b4f6d875a8a49d54a9e2ea51f90f6a5eda98f89c9a898db2d72f88a37a7f"
  },
  "results/validation_reported.json": {
    "base64": "ewogICJwcm92ZW5hbmNlIjogIlZhbHVlcyB0cmFuc2NyaWJlZCBmcm9tIHRoZSBzdXBwbGllZCBtYW51c2NyaXB0IGFuZCB0aGUgYXV0aG9yJ3MgYXVkaXQgcmVxdWVzdCwgMjUgT2N0b2JlciAyMDI2LiBObyByYXcgQXV0b0F0dGFjayBsb2dzLCBwZXItZXhhbXBsZSBwcmVkaWN0aW9ucywgc3Vic2V0IGluZGV4IG1hbmlmZXN0LCBvciBmdWxsLW1vZGVsIGNoZWNrcG9pbnQgZmlsZXMgd2VyZSBhdmFpbGFibGUgdG8gdGhlIGF1ZGl0b3IuIiwKICAic3RhdHVzIjogImF1dGhvcl9yZXBvcnRlZF9yb3VuZGVkX3BlcmNlbnRhZ2VzIiwKICAic3BsaXQiOiAidmFsaWRhdGlvbiIsCiAgInJlcG9ydGVkX3N1YnNldF9zaXplIjogNTEyLAogICJzYW1lX3N1YnNldF9hY3Jvc3NfbWV0aG9kcyI6ICJyZXBvcnRlZDsgaW5kZXggaWRlbnRpdHkgbm90IGluZGVwZW5kZW50bHkgdmVyaWZpZWQiLAogICJlcHNpbG9uIjogMC4zLAogICJwZ2RfYWxwaGEiOiAwLjAxLAogICJzZWVkcyI6IFswLCAxLCAyXSwKICAiYXR0YWNrcyI6IFsKICAgIHsibmFtZSI6ICJQR0QxMDAgeCAyMCIsICJhY2N1cmFjeV9wZXJjZW50IjogWzg0LjM4LCA4Ni41MiwgODMuOThdfSwKICAgIHsibmFtZSI6ICJGdWxsIHN0YW5kYXJkIEF1dG9BdHRhY2siLCAiYWNjdXJhY3lfcGVyY2VudCI6IFs4Mi42MiwgODUuMTYsIDgyLjIzXX0KICBdLAogICJhdXRvYXR0YWNrX2NvbXBvbmVudHNfcmVwb3J0ZWQiOiBbIkFQR0QtQ0UiLCAidGFyZ2V0ZWQgQVBHRCIsICJ0YXJnZXRlZCBGQUIiLCAiU3F1YXJlIEF0dGFjayJdLAogICJhZGRpdGlvbmFsX3NxdWFyZV9jaGVjayI6IHsic2VlZCI6IDAsICJiZWZvcmVfcGVyY2VudCI6IDgyLjYyLCAiYWZ0ZXJfcGVyY2VudCI6IDgyLjQyLCAiZXhhY3RfcXVlcnlfYW5kX3Jlc3RhcnRfYnVkZ2V0IjogbnVsbCwgInN0YXR1cyI6ICJhdXRob3JfcmVwb3J0ZWQ7IHJhdyBsb2cgdW5hdmFpbGFibGUifSwKICAiYWdncmVnYXRpb24iOiAiTWVhbiBhbmQgc2FtcGxlIFNEIChkZG9mPTEpIGFyZSByZWNvbXB1dGVkIGZyb20gdGhlIHN1cHBsaWVkIHJvdW5kZWQgcGVyY2VudGFnZXMuIFRoZXkgYXJlIG5vdCBpbmZlcnJlZCBleGFjdCBjb3VudHMgb3V0IG9mIDUxMi4iLAogICJ3YXJuaW5nIjogIkF1dGhvciByZXBvcnRzIGEgU3F1YXJlIEF0dGFjayB3YXJuaW5nLiBUaGVzZSBtZWFzdXJlbWVudHMgZG8gbm90IGVzdGFibGlzaCBhYnNlbmNlIG9mIGdyYWRpZW50IG1hc2tpbmcgb3IgY2VydGlmaWVkIHJvYnVzdG5lc3MuIERvIG5vdCBwb29sIHdpdGggdGhlIDEwMDAwLWltYWdlIGZpbmFsIHRlc3QuIgp9Cg==",
    "sha256": "4fd6e562e02a215d63249285c99ac38b9a55062da846fb4af655f3e999da7626"
  },
  "tools/verify_results.py": {
    "base64": "IiIiVmFsaWRhdGUgcHJlc2VydmVkIG1lYXN1cmVtZW50cyBhbmQgcmVnZW5lcmF0ZSByZXBvcnQgZmlndXJlcyB3aXRob3V0IHRyYWluaW5nLgoKSGlzdG9yaWNhbCBmaW5hbCB2YWx1ZXMgYXJlIG5ldmVyIGVkaXRlZC4gVmFsaWRhdGlvbiBmaWd1cmVzIHVzZSBleHBsaWNpdGx5CmxhYmVsbGVkIGF1dGhvci1yZXBvcnRlZCByb3VuZGVkIHBlcmNlbnRhZ2VzLCBub3QgaW52ZW50ZWQgcmF3IGF0dGFjayBsb2dzLgoiIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IHJlCmltcG9ydCBzdGF0aXN0aWNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdCk1FVEhPRFMgPSAoJ0Jhc2VsaW5lJywgJ1BHRDQwJywgJ0N1cnJpY3VsdW0nKQpNRVRSSUNTID0gKCdjbGVhbl9hY2N1cmFjeScsICdmZ3NtX2FjY3VyYWN5JywgJ3BnZF9hY2N1cmFjeScpCgoKZGVmIHZlcmlmeShyb290PVJPT1QsIG1ha2VfZmlndXJlcz1UcnVlKToKICAgIHJvb3QgPSBQYXRoKHJvb3QpCiAgICBzb3VyY2UgPSByb290IC8gJ3Jlc3VsdHMvRklOQUxfVEVTVF9BTExfTU9ERUxTLmNzdicKICAgIHByb3RvY29sID0ganNvbi5sb2Fkcygocm9vdCAvICdyZXN1bHRzL0ZJTkFMX1RFU1RfUFJPVE9DT0wuanNvbicpLnJlYWRfdGV4dCgpKQogICAgc291cmNlX2hhc2ggPSBoYXNobGliLnNoYTI1Nihzb3VyY2UucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKQogICAgaWYgc291cmNlX2hhc2ggIT0gcHJvdG9jb2xbJ21lYXN1cmVtZW50X3NoYTI1NiddOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ09yaWdpbmFsIGZpbmFsIG1lYXN1cmVtZW50IGJ5dGVzIGNoYW5nZWQ7IGludmVzdGlnYXRlIGJlZm9yZSB1cGRhdGluZyBwcm92ZW5hbmNlLicpCiAgICByZWNvcmRzID0gbGlzdChjc3YuRGljdFJlYWRlcihzb3VyY2Uub3BlbihuZXdsaW5lPScnKSkpCiAgICBpZiBsZW4ocmVjb3JkcykgIT0gOSBvciBzZXQoclsnbWV0aG9kJ10gZm9yIHIgaW4gcmVjb3JkcykgIT0gc2V0KE1FVEhPRFMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0V4cGVjdGVkIGV4YWN0bHkgbmluZSByZWNvcmRzIGFuZCBhbGwgdGhyZWUgbWV0aG9kcy4nKQogICAgZXhwZWN0ZWRfZXBvY2hzID0geydCYXNlbGluZSc6IFsxMCwgOCwgMTBdLCAnUEdENDAnOiBbMTAsIDEsIDFdLCAnQ3VycmljdWx1bSc6IFsxMCwgMTAsIDEwXX0KICAgIHN1bW1hcnkgPSBbXQogICAgZm9yIG1ldGhvZCBpbiBNRVRIT0RTOgogICAgICAgIGdyb3VwID0gc29ydGVkKChyIGZvciByIGluIHJlY29yZHMgaWYgclsnbWV0aG9kJ10gPT0gbWV0aG9kKSwga2V5PWxhbWJkYSByOiBpbnQoclsnc2VlZCddKSkKICAgICAgICBpZiBbaW50KHJbJ3NlZWQnXSkgZm9yIHIgaW4gZ3JvdXBdICE9IFswLCAxLCAyXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ0R1cGxpY2F0ZS9taXNzaW5nIHNlZWRzOiB7bWV0aG9kfScpCiAgICAgICAgaWYgW2ludChyWydiZXN0X2Vwb2NoJ10pIGZvciByIGluIGdyb3VwXSAhPSBleHBlY3RlZF9lcG9jaHNbbWV0aG9kXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ1NlbGVjdGVkIGVwb2NocyBkaWZmZXIgZnJvbSB0aGUgcHJlc2VydmVkIGhpc3RvcmljYWwgcmVjb3JkOiB7bWV0aG9kfScpCiAgICAgICAgZm9yIHJvdyBpbiBncm91cDoKICAgICAgICAgICAgaWYgaW50KHJvd1sndGVzdF9pbWFnZXMnXSkgIT0gMTAwMDAgb3Igbm90IHJlLmZ1bGxtYXRjaChyJ1swLTlhLWZdezY0fScsIHJvd1snY2hlY2twb2ludF9zaGEyNTYnXSk6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdJbnZhbGlkIGRlbm9taW5hdG9yIG9yIFNIQS0yNTYgc3ludGF4LicpCiAgICAgICAgICAgIGZvciBtZXRyaWMgaW4gTUVUUklDUzoKICAgICAgICAgICAgICAgIHZhbHVlID0gZmxvYXQocm93W21ldHJpY10pCiAgICAgICAgICAgICAgICBpZiBub3QgbWF0aC5pc2Zpbml0ZSh2YWx1ZSkgb3Igbm90IDAgPD0gdmFsdWUgPD0gMToKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnSW52YWxpZCBhY2N1cmFjeToge3Jvd30nKQogICAgICAgICAgICAgICAgaWYgYWJzKHZhbHVlICogMTAwMDAgLSByb3VuZCh2YWx1ZSAqIDEwMDAwKSkgPiAxZS04OgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0FjY3VyYWN5IGlzIG5vdCBhbiBpbnRlZ2VyIGNvdW50IG92ZXIgMTAwMDAuJykKICAgICAgICAgICAgaWYgZmxvYXQocm93WydwZ2RfYWNjdXJhY3knXSkgPiBmbG9hdChyb3dbJ2NsZWFuX2FjY3VyYWN5J10pIG9yIGZsb2F0KHJvd1snZmdzbV9hY2N1cmFjeSddKSA+IGZsb2F0KHJvd1snY2xlYW5fYWNjdXJhY3knXSk6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdBdHRhY2stc3Vydml2YWwgYWNjdXJhY3kgZXhjZWVkcyBjbGVhbiBhY2N1cmFjeS4nKQogICAgICAgIGZvciBtZXRyaWMgaW4gTUVUUklDUzoKICAgICAgICAgICAgdmFsdWVzID0gW2Zsb2F0KHJvd1ttZXRyaWNdKSAqIDEwMCBmb3Igcm93IGluIGdyb3VwXQogICAgICAgICAgICBzdW1tYXJ5LmFwcGVuZCh7J21ldGhvZCc6IG1ldGhvZCwgJ21ldHJpYyc6IG1ldHJpYywgJ25fc2VlZHMnOiAzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgJ21lYW5fcGVyY2VudCc6IHN0YXRpc3RpY3MubWVhbih2YWx1ZXMpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgJ3NhbXBsZV9zZF9wZXJjZW50Jzogc3RhdGlzdGljcy5zdGRldih2YWx1ZXMpfSkKICAgIHZhbGlkYXRpb24gPSBqc29uLmxvYWRzKChyb290IC8gJ3Jlc3VsdHMvdmFsaWRhdGlvbl9yZXBvcnRlZC5qc29uJykucmVhZF90ZXh0KCkpCiAgICB2YWxfc3VtbWFyeSA9IFtdCiAgICBmb3IgYXR0YWNrIGluIHZhbGlkYXRpb25bJ2F0dGFja3MnXToKICAgICAgICB2YWx1ZXMgPSBhdHRhY2tbJ2FjY3VyYWN5X3BlcmNlbnQnXQogICAgICAgIGlmIGxlbih2YWx1ZXMpICE9IDMgb3IgYW55KG5vdCAwIDw9IHggPD0gMTAwIGZvciB4IGluIHZhbHVlcyk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0ludmFsaWQgcmVwb3J0ZWQgdmFsaWRhdGlvbiBwZXJjZW50YWdlcy4nKQogICAgICAgIHZhbF9zdW1tYXJ5LmFwcGVuZCh7J2F0dGFjayc6IGF0dGFja1snbmFtZSddLCAnbWVhbl9wZXJjZW50Jzogc3RhdGlzdGljcy5tZWFuKHZhbHVlcyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAnc2FtcGxlX3NkX3BlcmNlbnQnOiBzdGF0aXN0aWNzLnN0ZGV2KHZhbHVlcyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAnc291cmNlX3ByZWNpc2lvbic6ICdhdXRob3ItcmVwb3J0ZWQgcm91bmRlZCBwZXJjZW50YWdlcyd9KQogICAgb3V0ID0gcm9vdCAvICdyZXN1bHRzL3ZlcmlmaWVkJwogICAgb3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggKG91dCAvICdmaW5hbF9zdW1tYXJ5LmNzdicpLm9wZW4oJ3cnLCBuZXdsaW5lPScnKSBhcyBmOgogICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKGYsIGZpZWxkbmFtZXM9bGlzdChzdW1tYXJ5WzBdKSkKICAgICAgICB3cml0ZXIud3JpdGVoZWFkZXIoKQogICAgICAgIHdyaXRlci53cml0ZXJvd3Moc3VtbWFyeSkKICAgIHJlc3VsdCA9IHsnZmluYWxfY3N2X3NoYTI1Nic6IHNvdXJjZV9oYXNoLCAnbl9yZWNvcmRzJzogOSwgJ25fc2VlZHNfZWFjaCc6IDMsCiAgICAgICAgICAgICAgJ2ZpbmFsX3Rlc3QnOiBzdW1tYXJ5LCAncmVwb3J0ZWRfdmFsaWRhdGlvbic6IHZhbF9zdW1tYXJ5LAogICAgICAgICAgICAgICdjaGVja3BvaW50X2hhc2hlcyc6ICdzeW50YXggdmVyaWZpZWQgb25seTsgY2hlY2twb2ludCBmaWxlcyB1bmF2YWlsYWJsZScsCiAgICAgICAgICAgICAgJ3ZhbGlkYXRpb25fcHJvdmVuYW5jZSc6IHZhbGlkYXRpb25bJ3N0YXR1cyddfQogICAgKG91dCAvICdzdGF0aXN0aWNzLmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzdWx0LCBpbmRlbnQ9MikgKyAnXG4nKQogICAgbGluZXMgPSBbJyMgSW5kZXBlbmRlbnRseSByZWNvbXB1dGVkIHN0YXRpc3RpY3MnLCAnJywKICAgICAgICAgICAgICdGaW5hbCBDU1YgdmFsdWVzIGFyZSBwcmVzZXJ2ZWQuIFBlcmNlbnRhZ2VzIGJlbG93IHVzZSBzYW1wbGUgU0QgKGRkb2Y9MSkgYWNyb3NzIHRocmVlIHRyYWluaW5nIHNlZWRzLicsICcnLAogICAgICAgICAgICAgJ3wgTWV0aG9kIHwgQ2xlYW4gKCUpIHwgRkdTTSAoJSkgfCBQR0QxMDAgeCA1ICglKSB8JywgJ3wtLS18LS0tOnwtLS06fC0tLTp8J10KICAgIGZvciBtZXRob2QgaW4gTUVUSE9EUzoKICAgICAgICBjZWxscyA9IFtmIntyWydtZWFuX3BlcmNlbnQnXTouMmZ9IMKxIHtyWydzYW1wbGVfc2RfcGVyY2VudCddOi4yZn0iIGZvciByIGluIHN1bW1hcnkgaWYgclsnbWV0aG9kJ10gPT0gbWV0aG9kXQogICAgICAgIGxpbmVzLmFwcGVuZCgnfCAnICsgbWV0aG9kICsgJyB8ICcgKyAnIHwgJy5qb2luKGNlbGxzKSArICcgfCcpCiAgICBsaW5lcyArPSBbJycsICdTZXBhcmF0ZSB2YWxpZGF0aW9uIHN1bW1hcmllcyB1c2Ugcm91bmRlZCBhdXRob3ItcmVwb3J0ZWQgcGVyY2VudGFnZXMsIG5vdCByYXcgY291bnRzOicsICcnXQogICAgZm9yIHJvdyBpbiB2YWxfc3VtbWFyeToKICAgICAgICBsaW5lcy5hcHBlbmQoZiItIHtyb3dbJ2F0dGFjayddfToge3Jvd1snbWVhbl9wZXJjZW50J106LjJmfSDCsSB7cm93WydzYW1wbGVfc2RfcGVyY2VudCddOi4yZn0lIChyZXBvcnRlZCBOPTUxMikuIikKICAgIGxpbmVzICs9IFsnJywgJ1RoZSBBdXRvQXR0YWNrIG1lYW4gaXMgODMuMzQlIHdoZW4gY2FsY3VsYXRlZCBmcm9tIHRoZSBwcm92aWRlZCByb3VuZGVkIHBlcmNlbnRhZ2VzLiAnCiAgICAgICAgICAgICAgJ0V4YWN0IHJhdyBjb3VudHMgY291bGQgY2hhbmdlIHRoZSBsYXN0IGRpc3BsYXllZCBkaWdpdDsgbm8gY291bnRzIHdlcmUgcmVjb25zdHJ1Y3RlZCBhcyBvYnNlcnZhdGlvbnMuJywKICAgICAgICAgICAgICAnJywgJ05vIGZ1bGwtbW9kZWwgY2hlY2twb2ludHMgb3IgcmF3IHZhbGlkYXRpb24gYXR0YWNrIGxvZ3Mgd2VyZSBhdmFpbGFibGUgZm9yIGluZGVwZW5kZW50IHJlcGxheS4nXQogICAgKG91dCAvICdTVU1NQVJZLm1kJykud3JpdGVfdGV4dCgnXG4nLmpvaW4obGluZXMpICsgJ1xuJykKICAgIGlmIG1ha2VfZmlndXJlczoKICAgICAgICBwbG90KHJvb3QsIHJlY29yZHMsIHN1bW1hcnksIHZhbGlkYXRpb24sIHZhbF9zdW1tYXJ5KQogICAgcHJpbnQoJ1xuJy5qb2luKGxpbmVzKSkKICAgIHJldHVybiByZXN1bHQKCgpkZWYgcGxvdChyb290LCByZWNvcmRzLCBzdW1tYXJ5LCB2YWxpZGF0aW9uLCB2YWxfc3VtbWFyeSk6CiAgICBpbXBvcnQgbWF0cGxvdGxpYgogICAgbWF0cGxvdGxpYi51c2UoJ0FnZycpCiAgICBpbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKICAgIHBsdC5yY1BhcmFtcy51cGRhdGUoeydmb250LmZhbWlseSc6ICdEZWphVnUgU2FucycsICdmb250LnNpemUnOiAxMCwKICAgICAgICAgICAgICAgICAgICAgICAgICdheGVzLnNwaW5lcy50b3AnOiBGYWxzZSwgJ2F4ZXMuc3BpbmVzLnJpZ2h0JzogRmFsc2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAncGRmLmZvbnR0eXBlJzogNDIsICdwcy5mb250dHlwZSc6IDQyLCAnc2F2ZWZpZy5kcGknOiAyMjB9KQogICAgZm9sZGVyID0gcm9vdCAvICdyZXBvcnQvZmlndXJlcycKICAgIGZvbGRlci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBjb2xvcnMgPSBbJyMzMTVmOGMnLCAnI2JiNWIzNycsICcjMjU4NTZiJ10KICAgIGxhYmVscyA9IFsnU3RhbmRhcmQgYmFzZWxpbmUnLCAnUEdENDAgZnJvbSBzY3JhdGNoJywgJ1BHRDEwIOKGkiBQR0Q0MCddCiAgICBkZWYgc2F2ZShmaWcsIG5hbWUpOgogICAgICAgIGZpZy5zYXZlZmlnKGZvbGRlciAvIChuYW1lICsgJy5wbmcnKSwgYmJveF9pbmNoZXM9J3RpZ2h0JykKICAgICAgICBmaWcuc2F2ZWZpZyhmb2xkZXIgLyAobmFtZSArICcucGRmJyksIGJib3hfaW5jaGVzPSd0aWdodCcsIG1ldGFkYXRhPXsnQ3JlYXRpb25EYXRlJzogTm9uZSwgJ01vZERhdGUnOiBOb25lfSkKICAgICAgICBwbHQuY2xvc2UoZmlnKQogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg4LjAsIDQuNCkpCiAgICB4ID0gbnAuYXJhbmdlKDMpCiAgICBmb3IgaSwgbWV0aG9kIGluIGVudW1lcmF0ZShNRVRIT0RTKToKICAgICAgICBzZWxlY3RlZCA9IFtyIGZvciByIGluIHN1bW1hcnkgaWYgclsnbWV0aG9kJ10gPT0gbWV0aG9kXQogICAgICAgIG1lYW5zID0gW3JbJ21lYW5fcGVyY2VudCddIGZvciByIGluIHNlbGVjdGVkXQogICAgICAgIHNkcyA9IFtyWydzYW1wbGVfc2RfcGVyY2VudCddIGZvciByIGluIHNlbGVjdGVkXQogICAgICAgIHhzID0geCArIChpIC0gMSkgKiAwLjIwCiAgICAgICAgYXguZXJyb3JiYXIoeHMsIG1lYW5zLCB5ZXJyPXNkcywgZm10PSdvJywgY2Fwc2l6ZT01LCBjb2xvcj1jb2xvcnNbaV0sIGxhYmVsPWxhYmVsc1tpXSwgbWFya2Vyc2l6ZT02LCBsdz0xLjQpCiAgICAgICAgZm9yIGosIG1ldHJpYyBpbiBlbnVtZXJhdGUoTUVUUklDUyk6CiAgICAgICAgICAgIHJhdyA9IFsxMDAgKiBmbG9hdChyW21ldHJpY10pIGZvciByIGluIHJlY29yZHMgaWYgclsnbWV0aG9kJ10gPT0gbWV0aG9kXQogICAgICAgICAgICBheC5zY2F0dGVyKFt4c1tqXSAtIC4wMzUsIHhzW2pdLCB4c1tqXSArIC4wMzVdLCByYXcsIHM9MTgsIGZhY2Vjb2xvcnM9J25vbmUnLCBlZGdlY29sb3JzPWNvbG9yc1tpXSwgYWxwaGE9LjcpCiAgICBheC5heGhsaW5lKDAsIGNvbG9yPScjOTk5OTk5JywgbHc9LjcpCiAgICBheC5zZXQoeHRpY2tzPXgsIHh0aWNrbGFiZWxzPVsnQ2xlYW4nLCAnRkdTTScsICdQR0QxMDAgw5cgNSddLCB5bGltPSgtMTMsIDEwNiksIHlsYWJlbD0nVGVzdCBhY2N1cmFjeSAoJSknKQogICAgZmlnLnN1cHRpdGxlKCdGdWxsIE1OSVNUIHRlc3Q6IDEwLDAwMCBpbWFnZXMgcGVyIGNoZWNrcG9pbnQnLCB5PS45OSwgZm9udHNpemU9MTIpCiAgICBheC5sZWdlbmQobG9jPSdsb3dlciBjZW50ZXInLCBiYm94X3RvX2FuY2hvcj0oLjUsIDEuMDEpLCBuY29sPTMsIGZvbnRzaXplPTksIGZyYW1lb249RmFsc2UpCiAgICBmaWcudGV4dCguMTAsIC4wMSwgJ1NvbGlkIHBvaW50czogbWVhbiDCsSBzYW1wbGUgU0QsIG49MyBzZWVkcy4gT3BlbiBwb2ludHM6IGluZGl2aWR1YWwgc2VlZHMuXG5TRCBpcyBkZXNjcmlwdGl2ZSwgbm90IGEgY29uZmlkZW5jZSBpbnRlcnZhbDsgYSBzeW1tZXRyaWMgU0QgcmFuZ2UgY2FuIGV4dGVuZCBiZWxvdyB6ZXJvLicsIGZvbnRzaXplPTgpCiAgICBmaWcudGlnaHRfbGF5b3V0KHJlY3Q9KDAsIC4xMCwgMSwgLjkyKSkKICAgIHNhdmUoZmlnLCAnZmluYWxfYWNjdXJhY3lfY29tcGFyaXNvbicpCiAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDcuMiwgMy45KSkKICAgIGZvciBpLCBtZXRob2QgaW4gZW51bWVyYXRlKE1FVEhPRFMpOgogICAgICAgIGdyb3VwID0gc29ydGVkKChyIGZvciByIGluIHJlY29yZHMgaWYgclsnbWV0aG9kJ10gPT0gbWV0aG9kKSwga2V5PWxhbWJkYSByOiBpbnQoclsnc2VlZCddKSkKICAgICAgICB5cyA9IFsxMDAgKiBmbG9hdChyWydwZ2RfYWNjdXJhY3knXSkgZm9yIHIgaW4gZ3JvdXBdCiAgICAgICAgeHMgPSBucC5hcmFuZ2UoMykgKyAoaSAtIDEpICogLjEyCiAgICAgICAgYXguc2NhdHRlcih4cywgeXMsIGNvbG9yPWNvbG9yc1tpXSwgbWFya2VyPVsnbycsICdzJywgJ14nXVtpXSwgcz02NSwgbGFiZWw9bGFiZWxzW2ldKQogICAgICAgIGZvciBweCwgcHkgaW4gemlwKHhzLCB5cyk6CiAgICAgICAgICAgIGF4LmFubm90YXRlKGYne3B5Oi4yZn0nLCAocHgsIHB5KSwgeHl0ZXh0PSgwLCA3KSwgdGV4dGNvb3Jkcz0nb2Zmc2V0IHBvaW50cycsIGhhPSdjZW50ZXInLCBmb250c2l6ZT04KQogICAgYXguc2V0KHh0aWNrcz1yYW5nZSgzKSwgeHRpY2tsYWJlbHM9WydTZWVkIDAnLCAnU2VlZCAxJywgJ1NlZWQgMiddLCB5bGltPSgtNSwgMTA0KSwgeWxhYmVsPSdQR0QxMDAgw5cgNSB0ZXN0IGFjY3VyYWN5ICglKScpCiAgICBheC5zZXRfdGl0bGUoJ0luZGVwZW5kZW50IHRyYWluaW5nIHNlZWRzOyBubyBmYWlsZWQgcnVucyBleGNsdWRlZCcpCiAgICBheC5sZWdlbmQobG9jPSdsb3dlciBsZWZ0JywgYmJveF90b19hbmNob3I9KDAsIDEuMDYpLCBuY29sPTMsIGZvbnRzaXplPTgsIGZyYW1lb249RmFsc2UpCiAgICBheC5ncmlkKGF4aXM9J3knLCBhbHBoYT0uMikKICAgIGZpZy50aWdodF9sYXlvdXQoKQogICAgc2F2ZShmaWcsICdzZWVkX3JvYnVzdG5lc3MnKQogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg3LjIsIDMuOSkpCiAgICBmb3IgaSwgYXR0YWNrIGluIGVudW1lcmF0ZSh2YWxpZGF0aW9uWydhdHRhY2tzJ10pOgogICAgICAgIHZhbHVlcyA9IGF0dGFja1snYWNjdXJhY3lfcGVyY2VudCddCiAgICAgICAgYXguc2NhdHRlcih2YWx1ZXMsIFsxLWktLjEwLCAxLWksIDEtaSsuMTBdLCBzPTI4LCBjb2xvcj1jb2xvcnNbaV0sIGFscGhhPS43KQogICAgICAgIHJvdyA9IHZhbF9zdW1tYXJ5W2ldCiAgICAgICAgYXguZXJyb3JiYXIocm93WydtZWFuX3BlcmNlbnQnXSwgMS1pLCB4ZXJyPXJvd1snc2FtcGxlX3NkX3BlcmNlbnQnXSwgZm10PSdEJywgY2Fwc2l6ZT01LCBtYXJrZXJzaXplPTcsIGNvbG9yPWNvbG9yc1tpXSkKICAgICAgICBheC50ZXh0KDg5LjQsIDEtaSwgZiJ7cm93WydtZWFuX3BlcmNlbnQnXTouMmZ9IMKxIHtyb3dbJ3NhbXBsZV9zZF9wZXJjZW50J106LjJmfSIsIHZhPSdjZW50ZXInLCBoYT0ncmlnaHQnLCBmb250c2l6ZT05KQogICAgYXguc2V0KHl0aWNrcz1bMSwwXSwgeXRpY2tsYWJlbHM9WydQR0QxMDAgw5cgMjAnLCAnRnVsbCBBdXRvQXR0YWNrJ10sIHhsaW09KDc5LDkwKSwgeWxpbT0oLS41LDEuNSksIHhsYWJlbD0nUmVwb3J0ZWQgdmFsaWRhdGlvbiBhY2N1cmFjeSAoJSkgKGV4cGFuZGVkIHNjYWxlKScpCiAgICBheC5zZXRfdGl0bGUoJ0N1cnJpY3VsdW06IHJlcG9ydGVkIDUxMi1pbWFnZSB2YWxpZGF0aW9uIHN1YnNldCcsIHBhZD0xMikKICAgIGF4LmdyaWQoYXhpcz0neCcsIGFscGhhPS4yKQogICAgZmlnLnRleHQoLjA0LC4wMSwnRGlhbW9uZHM6IG1lYW4gwrEgc2FtcGxlIFNEOyBzbWFsbCBwb2ludHM6IHNlZWRzIDDigJMyLiBSb3VuZGVkIGF1dGhvci1yZXBvcnRlZCB2YWx1ZXMuXG5SYXcgbG9ncyBhbmQgc3Vic2V0IGlkZW50aXRpZXMgd2VyZSB1bmF2YWlsYWJsZSBmb3IgaW5kZXBlbmRlbnQgdmVyaWZpY2F0aW9uLicsZm9udHNpemU9OCkKICAgIGZpZy50aWdodF9sYXlvdXQocmVjdD0oMCwuMTIsMSwxKSkKICAgIHNhdmUoZmlnLCd2YWxpZGF0aW9uX2F0dGFja19jb21wYXJpc29uJykKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tbm8tZmlndXJlcycsIGFjdGlvbj0nc3RvcmVfdHJ1ZScpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgdmVyaWZ5KG1ha2VfZmlndXJlcz1ub3QgYXJncy5ub19maWd1cmVzKQo=",
    "sha256": "b4a61f9a93e9d36bf039bd21ba09f94342537f8a5be5d15f75b13dac90c7600e"
  }
}
BUNDLE_SHA256 = 'c0ba0eb078ed437f1a9675a313667fc29790a53963b5e241d1a92ac37ca3cdb7'


In [ ]:
SNAPSHOT_DIR = ANALYSIS_ROOT / ("snapshot_" + BUNDLE_SHA256[:16])
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
for relative, item in EMBEDDED.items():
    raw = base64.b64decode(item["base64"], validate=True)
    assert hashlib.sha256(raw).hexdigest() == item["sha256"], relative
    destination = SNAPSHOT_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.read_bytes() != raw:
        raise RuntimeError(
            f"Mevcut kaynak değişmiş: {destination}. Dosya korunuyor; "
            "ANALYSIS_ROOT için yeni bir klasör seçip tekrar çalıştırın."
        )
    if not destination.exists():
        destination.write_bytes(raw)
print("Kaynak snapshot:", SNAPSHOT_DIR)
print("Kaynak paketi SHA-256:", BUNDLE_SHA256)
print("Dört gömülü kaynağın byte/hash denetimi geçti.")


## 3 · Kayıtları ve protokolün kanıt sınırlarını göster

CSV değerleri **0–1 aralığında oran** olarak saklanır; aşağıdaki kaynak tablosunda da
bu biçim korunur. Örneğin `0.975`, **%97.5** demektir; `975` değildir.
Tarihî eğitim veya saldırı tekrarını bu hücrelerden çıkarmayın.


In [ ]:
import csv
from IPython.display import Markdown, display

protocol = json.loads((SNAPSHOT_DIR / "results/FINAL_TEST_PROTOCOL.json").read_text())
validation = json.loads((SNAPSHOT_DIR / "results/validation_reported.json").read_text())
with (SNAPSHOT_DIR / "results/FINAL_TEST_ALL_MODELS.csv").open(newline="") as handle:
    recorded = list(csv.DictReader(handle))
columns = ["method", "seed", "best_epoch", "clean_accuracy", "fgsm_accuracy", "pgd_accuracy", "test_images"]
table = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
table += ["| " + " | ".join(row[column] for column in columns) + " |" for row in recorded]
display(Markdown("\n".join(table)))
print("Test protokolünün kaynağı:", protocol["protocol_provenance"])
print("Validation kaynağı:", validation["provenance"])
print("Validation durumu:", validation["status"])


## 4 · İstatistikleri ve yayın için şekilleri yeniden hesapla

Denetim; özgün CSV hash'ini, dokuz satırı, her yöntem için seed 0/1/2'yi,
seçili epoch'ları, 10.000 görüntülük paydaları, oran aralıklarını ve hash biçimini kontrol eder.
Ortalama ve **örnek standart sapma (ddof=1)** eğitim seed'leri üzerinden hesaplanır;
bu standart hata veya güven aralığı değildir. İki düşük doğruluklu PGD40 koşusu korunur.

Çıktılar `results/verified/` ve `report/figures/` altında oluşturulur.
Validation özetleri, raporlanmış yuvarlatılmış yüzdelerden hesaplanır. AutoAttack için
%83.34 ortalama bu girdilere aittir; mevcut olmayan kesin sayımlar üretilmez.


In [ ]:
environment = os.environ.copy()
environment.setdefault("MPLCONFIGDIR", str(SNAPSHOT_DIR / ".matplotlib_cache"))
completed = subprocess.run(
    [sys.executable, str(SNAPSHOT_DIR / "tools/verify_results.py")],
    cwd=SNAPSHOT_DIR, env=environment, text=True, capture_output=True,
)
if completed.returncode:
    print(completed.stdout)
    print(completed.stderr)
    completed.check_returncode()
VERIFIED_DIR = SNAPSHOT_DIR / "results/verified"
(VERIFIED_DIR / "analysis_log.txt").write_text(completed.stdout + completed.stderr)
display(Markdown((VERIFIED_DIR / "SUMMARY.md").read_text()))
statistics = json.loads((VERIFIED_DIR / "statistics.json").read_text())
assert statistics["n_records"] == 9 and statistics["n_seeds_each"] == 3
print("Aritmetik ve kaynak biçimi denetimi tamamlandı. Eğitim/saldırı yeniden çalıştırılmadı.")


## 5 · Şekilleri incele

İlk iki şekil kaydedilmiş **10.000 görüntülük test** sonuçlarını gösterir.
Son şekil, ham logları bulunmayan **512 görüntülük validation beyanlarını** gösterir;
genişletilmiş ölçek açıkça etiketlenir. Seed'ler bağımsız eğitim koşularıdır, zaman adımları değildir.
PNG ve vektörel PDF sürümleri birlikte kaydedilir.

Rapor/sunum için tablo ve şekillerin ekran görüntülerini de saklayın; asıl sayısal kaynak
CSV/JSON dosyalarıdır. Bu şekiller sertifikalı dayanıklılık kanıtı değildir.


In [ ]:
from IPython.display import Image

FIGURE_DIR = SNAPSHOT_DIR / "report/figures"
for stem in ("final_accuracy_comparison", "seed_robustness", "validation_attack_comparison"):
    display(Markdown("### " + stem.replace("_", " ")))
    display(Image(filename=str(FIGURE_DIR / (stem + ".png"))))


## 6 · Analiz çıktılarının ZIP kopyasını sakla

ZIP; gömülü kaynak kopyalarını, yeniden hesaplanan istatistikleri ve PNG/PDF şekillerini içerir.
**Checkpoint veya yeni saldırı sonucu içermez.** Kaggle'da dosyaları kalıcı tutmak için
**Save Version** kullanın. Colab otomatik indirmesi isteğe bağlıdır; varsayılan kapalıdır.


In [ ]:
import zipfile
from IPython.display import FileLink

archive = SNAPSHOT_DIR / "recorded_results_analysis.zip"
paths = sorted(
    path for folder in (SNAPSHOT_DIR / "results", FIGURE_DIR, SNAPSHOT_DIR / "tools")
    for path in folder.rglob("*") if path.is_file()
)
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as package:
    for path in paths:
        entry = zipfile.ZipInfo(path.relative_to(SNAPSHOT_DIR).as_posix(), date_time=(1980, 1, 1, 0, 0, 0))
        entry.compress_type = zipfile.ZIP_DEFLATED
        package.writestr(entry, path.read_bytes())
print("Çıktı arşivi:", archive)
display(FileLink(os.path.relpath(archive, Path.cwd())))

DOWNLOAD_ZIP = False
if DOWNLOAD_ZIP:
    try:
        from google.colab import files
    except ImportError:
        print("Otomatik indirme yalnız Colab içindir; üstteki bağlantıyı kullanın.")
    else:
        files.download(str(archive))
